In [22]:
import geopandas as gpd
import pandas as pd
import re
import os
from shapely import Point

#### Paths

In [5]:
names_dir = '../../../data/shapes'
farmers_xls = f'{names_dir}/farmers.xlsx'
shp_path = f'{names_dir}/awg_farmers_final.shp'
assert os.path.exists(farmers_xls), 'Check path'

#### Read in excel file and clean the headers

In [13]:
df = pd.read_excel(farmers_xls, skiprows=2, sheet_name='Sheet1')

In [19]:
df.head()

,names_of_famers,trees_given_,_estimanted_number_of\ntrees_per_acre,spacing_,species_,acreage,coordinates
0,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E"
1,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E"
2,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E"
3,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E"
4,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E"


In [17]:
df = df.iloc[:,4:]

In [18]:
df.columns = df.columns.str.replace(' ', '_').str.lower()

#### Convert the coordinates from deg min secs to decimal degrees

In [20]:
def parse_dms(dms_str):
    if pd.isna(dms_str): return None, None
    
    # Matches: degrees° minutes' seconds" direction
    pattern = r"(\d+(?:\.\d+)?)°(\d+(?:\.\d+)?)\'(\d+(?:\.\d+)?)\"([NSEW])"
    parts = re.findall(pattern, str(dms_str))
    
    def to_dd(deg, mn, sec, direction):
        dd = float(deg) + float(mn)/60 + float(sec)/3600
        if direction in ['S', 'W']:
            dd *= -1
        return dd

    # Returns (Latitude, Longitude) as floats
    if len(parts) == 2:
        lat = to_dd(*parts[0])
        lon = to_dd(*parts[1])
        return lat, lon
    return None, None

# Usage with your Excel file
df[['lat', 'lon']] = df['coordinates'].apply(lambda x: pd.Series(parse_dms(x)))

In [21]:
df.head()

,names_of_famers,trees_given_,_estimanted_number_of\ntrees_per_acre,spacing_,species_,acreage,coordinates,lat,lon
0,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E",-1.582333,37.865167
1,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E",-1.539500,37.880694
2,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E",-1.618972,37.860639
3,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E",-1.623250,37.871306
4,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E",-1.619444,37.869028


In [28]:
df.dropna(subset=['lat', 'lon'], inplace=True)

#### Create geometry column using shapely

In [29]:
geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]

In [30]:
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

In [31]:
gdf.head()

,names_of_famers,trees_given_,_estimanted_number_of\ntrees_per_acre,spacing_,species_,acreage,coordinates,lat,lon,geometry
0,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E",-1.582333,37.865167,POINT (37.86517 -1.58233)
1,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E",-1.539500,37.880694,POINT (37.88069 -1.5395)
2,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E",-1.618972,37.860639,POINT (37.86064 -1.61897)
3,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E",-1.623250,37.871306,POINT (37.87131 -1.62325)
4,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E",-1.619444,37.869028,POINT (37.86903 -1.61944)


In [32]:
gdf.isna().sum()

names_of_famers                          0
trees_given_                             0
_estimanted_number_of\ntrees_per_acre    0
spacing_                                 0
species_                                 0
acreage                                  0
coordinates                              0
lat                                      0
lon                                      0
geometry                                 0
dtype: int64

In [33]:
len(gdf)

163

In [34]:
gdf.to_file(f'{names_dir}/clean_file.fgb')

In [37]:
gdf['species_'].unique()

<StringArray>
['Melia volkensii', 'Moringa oleifera']
Length: 2, dtype: str